# Notes

Комментарий. В профиле на платформе fatsecret надо в вайтлист айпиадресов внести айпи своего приложения 

# Import

In [1]:
import os
import webbrowser
from getpass import getpass
from urllib.parse import parse_qs
from dotenv import load_dotenv
from requests_oauthlib import OAuth1Session
from datetime import date
import pandas as pd

c:\Users\Илья\Desktop\Саморазвитие\Бизнес (папка резервирования)\venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.0) or chardet (7.4.3)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(


# Аутентификация пользователя

In [2]:
load_dotenv()
CLIENT_ID = os.getenv('ConsumerKey')
CLIENT_SECRET = os.getenv('ConsumerSecret')

def parse_token(text):
    data = parse_qs(text)
    return (data.get('oauth_token', [None])[0],
            data.get('oauth_token_secret', [None])[0])

session = OAuth1Session(CLIENT_ID, CLIENT_SECRET, callback_uri='oob', signature_type='BODY')
resp = session.post('https://authentication.fatsecret.com/oauth/request_token')
resp.raise_for_status()
request_token, request_token_secret = parse_token(resp.text)

auth_url = f'https://authentication.fatsecret.com/oauth/authorize?oauth_token={request_token}'
print(f'Перейдите по ссылке и разрешите доступ:\n{auth_url}')
webbrowser.open(auth_url)

verifier = getpass('Введите PIN-код: ').strip()

access_session = OAuth1Session(
    CLIENT_ID, CLIENT_SECRET,
    resource_owner_key=request_token,
    resource_owner_secret=request_token_secret,
    verifier=verifier,
    signature_type='BODY'
)
resp = access_session.post('https://authentication.fatsecret.com/oauth/access_token')
resp.raise_for_status()
access_token, access_token_secret = parse_token(resp.text)

api_session = OAuth1Session(
    CLIENT_ID, CLIENT_SECRET,
    resource_owner_key=access_token,
    resource_owner_secret=access_token_secret
)


Перейдите по ссылке и разрешите доступ:
https://authentication.fatsecret.com/oauth/authorize?oauth_token=749100e3590b46e58eab6806094f4b45


# Данные

## Пользователь

In [ ]:
response = api_session.get(
    'https://platform.fatsecret.com/rest/server.api',
    params={'method': 'profile.get', 'format': 'json'}
)
response.raise_for_status()

# response.json()

# {'profile': {'goal_weight_kg': '92.0000',
#   'height_cm': '185.00',
#   'height_measure': 'Cm',
#   'last_weight_date_int': '20507',
#   'last_weight_kg': '87.5000',
#   'weight_measure': 'Kg'}}

## Дневник питания

In [ ]:
from datetime import date, timedelta
import time

def date_to_int(d):
    return (d - date(1970, 1, 1)).days

start_str = '2026-04-01'
end_str = '2026-04-10'

start_date = date.fromisoformat(start_str)
end_date = date.fromisoformat(end_str)

frames = []
d = start_date
while d <= end_date:
    resp = api_session.get(
        'https://platform.fatsecret.com/rest/server.api',
        params={'method': 'food_entries.get.v2', 
                'format': 'json', 
                'date': date_to_int(d)}
    )
    resp.raise_for_status()
    frames.append(pd.DataFrame(resp.json()['food_entries']['food_entry']))
    d += timedelta(days=1)

    time.sleep(1)

df = pd.concat(frames, ignore_index=True)

def int_to_date_str(n):
    return pd.to_datetime(
        (date(1970, 1, 1) + timedelta(days=int(n))).strftime('%Y-%m-%d')
    )

df['date'] = df['date_int'].apply(int_to_date_str)
df.drop(columns=['date_int'], inplace=True, errors='ignore')

In [29]:
# df.columns

# ['calories', 'carbohydrate', 'cholesterol', 'fat', 'fiber',
    #    'food_entry_description', 'food_entry_id', 'food_entry_name', 'food_id',
    #    'meal', 'monounsaturated_fat', 'number_of_units', 'polyunsaturated_fat',
    #    'potassium', 'protein', 'saturated_fat', 'serving_id', 'sodium',
    #    'sugar', 'calcium', 'iron', 'vitamin_a', 'vitamin_c', 'date']

# df.info()

# <class 'pandas.core.frame.DataFrame'>
# RangeIndex: 103 entries, 0 to 102
# Data columns (total 24 columns):
#  #   Column                  Non-Null Count  Dtype         
# ---  ------                  --------------  -----         
#  0   calories                103 non-null    object        
#  1   carbohydrate            103 non-null    object        
#  2   cholesterol             21 non-null     object        
#  3   fat                     103 non-null    object        
#  4   fiber                   24 non-null     object        
#  5   food_entry_description  103 non-null    object        
#  6   food_entry_id           103 non-null    object        
#  7   food_entry_name         103 non-null    object        
#  8   food_id                 103 non-null    object        
#  9   meal                    103 non-null    object        
#  10  monounsaturated_fat     14 non-null     object        
#  11  number_of_units         103 non-null    object        
#  12  polyunsaturated_fat     14 non-null     object        
#  13  potassium               22 non-null     object        
#  14  protein                 103 non-null    object        
#  15  saturated_fat           27 non-null     object        
#  16  serving_id              103 non-null    object        
#  17  sodium                  38 non-null     object        
#  18  sugar                   50 non-null     object        
#  19  calcium                 6 non-null      object        
#  20  iron                    6 non-null      object        
#  21  vitamin_a               6 non-null      object        
#  22  vitamin_c               6 non-null      object        
#  23  date                    103 non-null    datetime64[ns]
# dtypes: datetime64[ns](1), object(23)
# memory usage: 19.4+ KB

## Наиболее употребляемая

In [31]:
response = api_session.get(
    'https://platform.fatsecret.com/rest/server.api',
    params={'method': 'foods.get_most_eaten.v2', 
            'format': 'json'}
)
response.raise_for_status()

In [ ]:
# response.json()

# {'foods': {'food': [{'brand_name': 'Савушкин Продукт',
#     'food_id': '17400944',
#     'food_name': 'Йогурт Греческий Teos 2%',
#     'food_type': 'Brand',
#     'food_url': 'https://foods.fatsecret.com/calories-nutrition/ru/%D0%A1%D0%B0%D0%B2%D1%83%D1%88%D0%BA%D0%B8%D0%BD-%D0%9F%D1%80%D0%BE%D0%B4%D1%83%D0%BA%D1%82/%D0%99%D0%BE%D0%B3%D1%83%D1%80%D1%82-%D0%93%D1%80%D0%B5%D1%87%D0%B5%D1%81%D0%BA%D0%B8%D0%B9-teos-2/100%D0%B3',
#     'number_of_units': '2.500',
#     'serving_id': '16411429'},
#    {'brand_name': 'Просто Молоко',
#     'food_id': '17052724',
#     'food_name': 'Молоко 1,5%',
#     'food_type': 'Brand',
#     'food_url': 'https://foods.fatsecret.com/calories-nutrition/ru/%D0%9F%D1%80%D0%BE%D1%81%D1%82%D0%BE-%D0%9C%D0%BE%D0%BB%D0%BE%D0%BA%D0%BE/%D0%9C%D0%BE%D0%BB%D0%BE%D0%BA%D0%BE-15/100%D0%BC%D0%BB',
#     'number_of_units': '2.190',
#     'serving_id': '16088703'},...]}}

## Недавно употребленная

In [32]:
response = api_session.get(
    'https://platform.fatsecret.com/rest/server.api',
    params={'method': 'foods.get_recently_eaten.v2', 
            'format': 'json'}
)
response.raise_for_status()

In [34]:
# response.json()

# {'foods': {'food': [{'brand_name': 'Коломенское',
#     'food_id': '106601888',
#     'food_name': 'Зефир Коломенский со Вкусом Ванили и Малины',
#     'food_type': 'Brand',
#     'food_url': 'https://foods.fatsecret.com/calories-nutrition/ru/%D0%9A%D0%BE%D0%BB%D0%BE%D0%BC%D0%B5%D0%BD%D1%81%D0%BA%D0%BE%D0%B5/%D0%97%D0%B5%D1%84%D0%B8%D1%80-%D0%9A%D0%BE%D0%BB%D0%BE%D0%BC%D0%B5%D0%BD%D1%81%D0%BA%D0%B8%D0%B9-%D1%81%D0%BE-%D0%92%D0%BA%D1%83%D1%81%D0%BE%D0%BC-%D0%92%D0%B0%D0%BD%D0%B8%D0%BB%D0%B8-%D0%B8-%D0%9C%D0%B0%D0%BB%D0%B8%D0%BD%D1%8B/100%D0%B3',
#     'number_of_units': '0.800',
#     'serving_id': '84683703'},...]}}

##  Сохраненные блюда

In [35]:
response = api_session.get(
    'https://platform.fatsecret.com/rest/server.api',
    params={'method': 'saved_meals.get.v2', 
            'format': 'json'}
)
response.raise_for_status()

In [ ]:
# response.json()

# {'saved_meals': {'saved_meal': [{'meals': 'Breakfast,Lunch,Dinner,Other',
#     'saved_meal_description': None,
#     'saved_meal_id': '46183002',
#     'saved_meal_name': 'Банан'},...]}}

## Упражнения константы

In [36]:
response = api_session.get(
    'https://platform.fatsecret.com/rest/server.api',
    params={'method': 'exercises.get.v2', 
            'format': 'json'}
)
response.raise_for_status()

In [ ]:
# response.json()

# {'exercises': {'exercise': [{'exercise_id': '0', 'exercise_name': 'Other'},
#    {'exercise_id': '77', 'exercise_name': 'Abdominal (Sit Ups)'},...]}}

## Упражнения за дату + калории

In [37]:
response = api_session.get(
    'https://platform.fatsecret.com/rest/server.api',
    params={'method': 'exercise_entries.get.v2', 
            'format': 'json',
            'date':start_date}
)
response.raise_for_status()

In [ ]:
# response.json()

# {'exercise_entries': {'exercise_entry': [{'calories': '1470',
#     'exercise_id': '2',
#     'exercise_name': 'Resting',
#     'is_template_value': '1',
#     'minutes': '960'},
#    {'calories': '662',
#     'exercise_id': '1',
#     'exercise_name': 'Sleeping',
#     'is_template_value': '1',
#     'minutes': '480'}]}}